<img src="https://upload.wikimedia.org/wikipedia/commons/3/35/Uba_fiuba_ingenieria_logo.png" width="300" align="center">



# **Analisis de Series de Tiempo II**

# **Clase 1, Leakage Temporal y Validacion Temporal**

Importamos lo necesario

In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold, TimeSeriesSplit

### **A) Leakage por Escalado**

Generamos una serie de juguete. El train es estable (~12). El test trae un outlier (100) que en produccion deberia detectarse como anomalia.

In [ ]:
X_train = np.array([10.0, 12.0, 14.0]).reshape(-1, 1)
X_test = np.array([100.0]).reshape(-1, 1)

####  **A.1) Incorrecto**

El scaler ve todo el dataset (train + test). Al hacer fit sobre la concatenacion, mu y sigma se calculan incluyendo el 100 futuro. El train queda normalizado "sabiendo" que ese outlier existe, eso es informacion del futuro filtrada al pasado.

In [ ]:
X_todo = np.vstack([X_train, X_test]) # concatena train + test
scaler_mal = StandardScaler().fit(X_todo) # fit usa el futuro (MAL)
z_test_mal = scaler_mal.transform(X_test)[0, 0]

####  **A.2) Correcto**

El scaler aprende solo del pasado (train + test). El fit solo en train, el test se transforma con esos mismos mu, sigma.

In [ ]:
scaler_bien = StandardScaler().fit(X_train) # fit SOLO en train
z_test_bien = scaler_bien.transform(X_test)[0, 0]

Comparamos los parametros aprendidos y el z-score del punto anomalo.

In [ ]:
print(f"Parametros del scaler:")
print(f"  MAL  (fit train+test): mu={scaler_mal.mean_[0]:.2f}  "
          f"sigma={scaler_mal.scale_[0]:.2f}")
print(f"  BIEN (fit solo train): mu={scaler_bien.mean_[0]:.2f}  "
          f"sigma={scaler_bien.scale_[0]:.2f}")
print(f"\nz-score del valor de test = 100:")
print(f"  MAL : z = {z_test_mal:5.2f}  -> el outlier queda 'aplastado', "
          f"parece casi normal")
print(f"  BIEN: z = {z_test_bien:5.2f}  -> el outlier se ve enorme, "
          f"como corresponde")

Parametros del scaler:
  MAL  (fit train+test): mu=34.00  sigma=38.13
  BIEN (fit solo train): mu=12.00  sigma=1.63

z-score del valor de test = 100:
  MAL : z =  1.73  -> el outlier queda 'aplastado', parece casi normal
  BIEN: z = 53.89  -> el outlier se ve enorme, como corresponde


Con leakage la metrica de validacion sale OPTIMISTA y luego no se reproduce en produccion, porque en produccion el scaler nunca podra haber visto el 100 antes de tiempo.

### **B) Leakage por Engineered Features**

Una feature en t SOLO puede usar informacion <= t.   

f_t = g(x_1, ..., x_t)


Errores tipicos: media movil centrada y lag invertido (shift negativo).

Definimos el dataset

In [ ]:
df = pd.DataFrame({"x": [10, 11, 12, 13, 14, 15, 16]})

####  **B.1) Incorrecto**

Ventana centrada. ma_centrada[t] usa t-1, t, t+1 mira el futuro. Tambien el lag invertido shift(-1) trae el futuro.

In [ ]:
df["ma_centrada"] = df["x"].rolling(window=3, center=True).mean()
df["lag_invertido"] = df["x"].shift(-1) # = x[t+1]  (MAL)

####  **B.2) Correcto**

Feature correcta (causal): ventana hacia atras. ma_trailing[t] usa t-2..t

In [ ]:
df["ma_trailing"] = df["x"].rolling(window=3).mean()
df["lag_correcto"] = df["x"].shift(1) # = x[t-1]  (BIEN)

In [ ]:
print(df.to_string(index=True))

    x  ma_centrada  lag_invertido  ma_trailing  lag_correcto
0  10          NaN           11.0          NaN           NaN
1  11         11.0           12.0          NaN          10.0
2  12         12.0           13.0         11.0          11.0
3  13         13.0           14.0         12.0          12.0
4  14         14.0           15.0         13.0          13.0
5  15         15.0           16.0         14.0          14.0
6  16          NaN            NaN         15.0          15.0


Demostracion del leakage

Prueba: si una feature en el instante t depende del FUTURO, entonces cambiar x en t+1 deberia cambiar el valor de la feature en t.

In [ ]:
t = 2 # instante a inspeccionar
ma_centrada_antes = df["x"].rolling(3, center=True).mean().iloc[t]
ma_trailing_antes = df["x"].rolling(3).mean().iloc[t]

In [ ]:
df_mod = df.copy()
df_mod.loc[t + 1, "x"] = 999                    # alteramos SOLO el futuro

In [ ]:
ma_centrada_desp = df_mod["x"].rolling(3, center=True).mean().iloc[t]
ma_trailing_desp = df_mod["x"].rolling(3).mean().iloc[t]

In [ ]:
print(f"\nAltero x en t+1 (futuro) y miro la feature en t={t}:")
print(f"  ma_centrada[t]:  {ma_centrada_antes} -> {ma_centrada_desp}  "
          f"CAMBIO => usa el futuro (LEAKAGE)")
print(f"  ma_trailing[t]:  {ma_trailing_antes} -> {ma_trailing_desp}  "
          f"NO cambia => solo usa el pasado (OK)")


Altero x en t+1 (futuro) y miro la feature en t=2:
  ma_centrada[t]:  12.0 -> 340.6666666666667  CAMBIO => usa el futuro (LEAKAGE)
  ma_trailing[t]:  11.0 -> 11.0  NO cambia => solo usa el pasado (OK)


Aun usando ventana trailing, si calculas la feature antes de separar los folds, la ventana cruza la frontera train/test. El feature engineering tambien debe respetar el corte temporal.

### **C) Leakage por Target Combination**

El target (o algo derivado de el) se cuela entre las features. Caso sutil con ventanas: la misma observacion es input de una muestra de train y target de otra de test. Defensa: purge (sacar solapadas) + embargo (gap entre train/test)

In [ ]:
serie = np.arange(0, 20) # indices 0..19 (cada numero = su timestamp)
L = 3 # lookback: la ventana de input mide 3
H = 2 # horizonte: el target mide 2

Construye muestras supervisadas con ventana deslizante.

Cada muestra guarda los rangos de indices que ocupa en el tiempo:
* input  = [t-L, t-1]   (lo que el modelo ve)
* target = [t,   t+H-1] (lo que debe predecir)

In [ ]:
muestras = []

for t in range(L, len(serie) - H + 1):
  muestras.append({
            "in_ini": t - L, "in_fin": t - 1, # rango del input
            "out_ini": t, "out_fin": t + H - 1, # rango del target
        })

Split temporal por muestra: las primeras a train, las ultimas a test

In [ ]:
corte = int(len(muestras) * 0.7)
train = muestras[:corte]
test = muestras[corte:]

El test arranca en este instante. Cualquier muestra de train cuyo target llegue a >= test_ini esta "viendo" tiempo que pertenece al test.

In [ ]:
test_ini = test[0]["in_ini"]

####  **C.1) Incorrecto**

Sin defensa: detectamos solapamiento

In [ ]:
solapadas = [m for m in train if m["out_fin"] >= test_ini]

In [ ]:
print(f"Test empieza en el instante {test_ini}.")
print(f"Muestras de train cuyo target invade el test (leakage): " f"{len(solapadas)}")
for m in solapadas:
  print(f"  input[{m['in_ini']}..{m['in_fin']}] " f"target[{m['out_ini']}..{m['out_fin']}], toca el test")

Test empieza en el instante 11.
Muestras de train cuyo target invade el test (leakage): 4
  input[7..9] target[10..11], toca el test
  input[8..10] target[11..12], toca el test
  input[9..11] target[12..13], toca el test
  input[10..12] target[13..14], toca el test


####  **C.2) Correcto**

Con defensa: purge + embargo


Con purge descartamos del train toda muestra cuyo target se solape con el inicio del test.

Embargo: ademas dejamos un gap de seguridad para cortar la autocorrelacion entre el fin del train y el test.

In [ ]:
embargo = H # gap >= horizonte
train_limpio = [m for m in train if m["out_fin"] < test_ini - embargo]

In [ ]:
print(f"\nCon purge + embargo (gap={embargo}):")
print(f"  train original: {len(train)} muestras")
print(f"  train limpio  : {len(train_limpio)} muestras "
          f"(se eliminaron las que tocaban el futuro)")


Con purge + embargo (gap=2):
  train original: 11 muestras
  train limpio  : 5 muestras (se eliminaron las que tocaban el futuro)


En sklearn esto se logra directo con TimeSeriesSplit (gap=...)

### **D) Holdout Temporal**

Split mas simple que respeta el orden: train = pasado, test = futuro, sin shuffle. Barajar (KFold shuffle) mete el futuro en el train, leakage.

In [ ]:
n = 10
X = np.arange(n).reshape(-1, 1) # indice = timestamp (0 = mas antiguo)

####  **D.1) Incorrecto**

**K-fold aleatorio**

Al barajar, un fold de test puede contener instantes anteriores a otros que quedaron en train, entrenamos con el futuro para predecir el pasado.

In [ ]:
kf = KFold(n_splits=2, shuffle=True, random_state=0)

for i, (tr, te) in enumerate(kf.split(X)):
  hay_futuro_en_train = tr.max() > te.min()

  print(f" fold {i}: train={sorted(tr.tolist())} test={sorted(te.tolist())}")
  print(f" max(train)={tr.max()} > min(test)={te.min()} ? "
              f"{hay_futuro_en_train}, True = leakage")


 fold 0: train=[0, 3, 5, 6, 7] test=[1, 2, 4, 8, 9]
 max(train)=7 > min(test)=1 ? True, True = leakage
 fold 1: train=[1, 2, 4, 8, 9] test=[0, 3, 5, 6, 7]
 max(train)=9 > min(test)=0 ? True, True = leakage


####  **D.2) Correcto**

Holdout que respeta el orden

In [ ]:
corte = int(n * 0.8)
train_idx, test_idx = np.arange(corte), np.arange(corte, n)

In [ ]:
print(f"Holdout temporal (sin shuffle):")
print(f"  train = {train_idx.tolist()}  (todo pasado)")
print(f"  test  = {test_idx.tolist()}  (todo futuro)")
print(f"  Regla cumplida: max(train)={train_idx.max()} < "
          f"min(test)={test_idx.min()}")

Holdout temporal (sin shuffle):
  train = [0, 1, 2, 3, 4, 5, 6, 7]  (todo pasado)
  test  = [8, 9]  (todo futuro)
  Regla cumplida: max(train)=7 < min(test)=8


Limitacion del holdout: una sola estimacion -> alta varianza y el test cubre un unico periodo que puede ser atipico.

### **E) Walk-Forward (rolling-origin, ventana fija)**

Avanzamos el origen y reevaluamos en varios folds. El train es una ventana de tamano fijo que se desliza (max_train_size). gap = embargo

In [ ]:
n = 12
X = np.arange(n).reshape(-1, 1)

* max_train_size fija el tamano del train, ventana deslizante (rolling).

* gap deja un hueco (embargo) entre el fin del train y el inicio del test.

In [ ]:
tscv = TimeSeriesSplit(n_splits=4, max_train_size=4, gap=1)

In [ ]:
for i, (tr, te) in enumerate(tscv.split(X)):
  # En cada fold el train SIEMPRE es anterior al test (no hay leakage)
  print(f"  fold {i}: train={tr.tolist()} (size={len(tr)})" f"test={te.tolist()}")

  fold 0: train=[0, 1, 2] (size=3)test=[4, 5]
  fold 1: train=[1, 2, 3, 4] (size=4)test=[6, 7]
  fold 2: train=[3, 4, 5, 6] (size=4)test=[8, 9]
  fold 3: train=[5, 6, 7, 8] (size=4)test=[10, 11]


Notar que el train queda acotado por max_train_size=4 (el fold 0 es menor solo porque aun no hay historia suficiente. La ventana se desliza y entre train y test queda 1 paso de embargo por el gap.

### **F) Expanding Window**

Variante de walk-forward donde el train crece en cada fold (no descarta el pasado). Es el comportamiento por defecto de TimeSeriesSplit.

In [ ]:
n = 12
X = np.arange(n).reshape(-1, 1)

Sin max_train_size, el train arranca en 0 y crece fold a fold

In [ ]:
tscv = TimeSeriesSplit(n_splits=4)

for i, (tr, te) in enumerate(tscv.split(X)):
  print(f"  fold {i}: train={tr.tolist()} (size={len(tr)})" f"test={te.tolist()}")

  fold 0: train=[0, 1, 2, 3] (size=4)test=[4, 5]
  fold 1: train=[0, 1, 2, 3, 4, 5] (size=6)test=[6, 7]
  fold 2: train=[0, 1, 2, 3, 4, 5, 6, 7] (size=8)test=[8, 9]
  fold 3: train=[0, 1, 2, 3, 4, 5, 6, 7, 8, 9] (size=10)test=[10, 11]


Rolling vs Expanding:
* Expanding: usa todo el pasado. Mejor si los patrones viejos" siguen valiendo (mas datos = mejor).")

* Rolling: ventana reciente fija. Mejor si hay concept drift, porque el pasado lejano ya no representa el presente.